In [1]:
import cv2
import mediapipe as mp
import os
import csv
import numpy as np
import random

# ================================
# PATH DATASET
# ================================
DATASET_PATH = "alfabet_bisindo"
IMAGE_ROOT = os.path.join(DATASET_PATH, "images")
SPLITS = ["train", "val"]
OUTPUT_CSV = "bisindo_alfabet_final.csv"

# ================================
# MEDIAPIPE HANDS
# ================================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.6
)

# ================================
# AUGMENTASI
# ================================
def augment_image(img):
    h, w = img.shape[:2]
    aug = img.copy()

    alpha = random.uniform(0.9, 1.1)
    beta = random.randint(-15, 15)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    return aug

# ================================
# EKSTRAK 1 TANGAN
# ================================
def extract_one_hand(hand_landmarks):
    lm = hand_landmarks.landmark
    pts = np.array([[p.x, p.y] for p in lm])

    base = pts[0]
    pts = pts - base

    max_val = np.max(np.abs(pts))
    if max_val != 0:
        pts = pts / max_val

    data = pts.flatten().tolist()

    def dist(a, b):
        return ((lm[a].x - lm[b].x) ** 2 + (lm[a].y - lm[b].y) ** 2) ** 0.5

    data.append(dist(4, 8))
    data.append(dist(8, 12))
    data.append(dist(12, 16))
    data.append(dist(16, 20))

    return data

# ================================
# EKSTRAK 2 TANGAN
# ================================
def extract_landmarks(rgb_img):
    result = hands.process(rgb_img)

    if not result.multi_hand_landmarks:
        return None

    empty_hand = [0.0] * 46
    left_data = empty_hand.copy()
    right_data = empty_hand.copy()

    for hand_landmarks, handedness in zip(
        result.multi_hand_landmarks,
        result.multi_handedness
    ):
        hand_label = handedness.classification[0].label

        if hand_label == "Left":
            left_data = extract_one_hand(hand_landmarks)
        elif hand_label == "Right":
            right_data = extract_one_hand(hand_landmarks)

    data = left_data + right_data

    if len(data) != 92:
        return None

    return data

# ================================
# PROSES DATASET
# ================================
rows = []
label_counts = {}

for split in SPLITS:
    split_path = os.path.join(IMAGE_ROOT, split)

    if not os.path.exists(split_path):
        print("Folder tidak ditemukan:", split_path)
        continue

    print(f"\nMemproses split: {split}")

    for label in sorted(os.listdir(split_path)):
        class_folder = os.path.join(split_path, label)

        if not os.path.isdir(class_folder):
            continue

        print(f"Class: {label}")

        for img_file in os.listdir(class_folder):
            if not img_file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue

            img_path = os.path.join(class_folder, img_file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (640, 480))

            # gambar asli
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            data = extract_landmarks(rgb)

            if data:
                rows.append(data + [label])
                label_counts[label] = label_counts.get(label, 0) + 1

            # gambar augmentasi
            aug = augment_image(img)
            rgb_aug = cv2.cvtColor(aug, cv2.COLOR_BGR2RGB)
            data_aug = extract_landmarks(rgb_aug)

            if data_aug:
                rows.append(data_aug + [label])
                label_counts[label] = label_counts.get(label, 0) + 1

# ================================
# SIMPAN CSV
# ================================
header = [f"f{i}" for i in range(92)] + ["label"]

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

hands.close()

print("\n✅ selesai:", OUTPUT_CSV)
print("Total data:", len(rows))
print("Distribusi label:")
for label, count in sorted(label_counts.items()):
    print(label, ":", count)


Memproses split: train
Class: A
Class: B
Class: C
Class: D
Class: E
Class: F
Class: G
Class: H
Class: I
Class: J
Class: K
Class: L
Class: M
Class: N
Class: O
Class: P
Class: Q
Class: R
Class: S
Class: T
Class: U
Class: V
Class: W
Class: X
Class: Y
Class: Z

Memproses split: val
Class: A
Class: B
Class: C
Class: D
Class: E
Class: F
Class: G
Class: H
Class: I
Class: J
Class: K
Class: L
Class: M
Class: N
Class: O
Class: P
Class: Q
Class: R
Class: S
Class: T
Class: U
Class: V
Class: W
Class: X
Class: Y
Class: Z

✅ selesai: bisindo_alfabet_final.csv
Total data: 19447
Distribusi label:
A : 566
B : 618
C : 848
D : 787
E : 862
F : 676
G : 809
H : 688
I : 829
J : 819
K : 656
L : 734
M : 887
N : 874
O : 884
P : 857
Q : 649
R : 763
S : 801
T : 679
U : 790
V : 808
W : 680
X : 727
Y : 499
Z : 657


In [1]:
# training_alfabet_xgb.py

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import joblib

# ================================
# LOAD CSV ALFABET
# ================================
FILE  = "bisindo_alfabet_final.csv"
MODEL = "model_bisindo_alfabet.pkl"

df = pd.read_csv(FILE)

X = df.drop("label", axis=1).values
y = df["label"].values

print(f"Total data: {len(df)}")
print(f"Jumlah kelas: {len(set(y))}")
print(f"Jumlah fitur: {X.shape[1]}")

print("\nDistribusi label:")
print(df["label"].value_counts())

# ================================
# LABEL ENCODER
# XGBoost lebih aman pakai angka
# ================================
le = LabelEncoder()
y  = le.fit_transform(y)

# ================================
# SPLIT DATA
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ================================
# TRAINING XGBOOST
# ================================
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ================================
# EVALUASI
# ================================
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print(f"\nAkurasi: {acc * 100:.2f}%")

# ubah balik ke label asli
y_test_str = le.inverse_transform(y_test)
pred_str   = le.inverse_transform(pred)

print("\nClassification Report:")
print(classification_report(y_test_str, pred_str))

# ================================
# SIMPAN MODEL
# ================================
joblib.dump(model, MODEL)
joblib.dump(le, "label_encoder_alfabet.pkl")

print(f"\n✅ Model tersimpan: {MODEL}")
print("✅ Label encoder tersimpan")

Total data: 19447
Jumlah kelas: 26
Jumlah fitur: 92

Distribusi label:
label
M    887
O    884
N    874
E    862
P    857
C    848
I    829
J    819
G    809
V    808
S    801
U    790
D    787
R    763
L    734
X    727
H    688
W    680
T    679
F    676
Z    657
K    656
Q    649
B    618
A    566
Y    499
Name: count, dtype: int64

Akurasi: 97.99%

Classification Report:
              precision    recall  f1-score   support

           A       0.97      0.96      0.97       113
           B       1.00      0.98      0.99       124
           C       0.98      0.99      0.98       170
           D       0.97      0.96      0.97       157
           E       0.96      1.00      0.98       172
           F       0.99      0.98      0.98       135
           G       0.99      1.00      0.99       162
           H       0.97      1.00      0.98       138
           I       1.00      1.00      1.00       166
           J       0.99      0.97      0.98       164
           K       0.98    